# 01 · Claude Agent SDK: una palanca por celda
Modelo `claude-sonnet-5`. Pregunta fija: *¿Cómo cambió el margen por proyecto de junio a julio, y por qué?*. Cada celda añade una opción de `ClaudeAgentOptions`.

Equivalencia con `pi`: `tools=[]` ≈ `-nt` · `skills=[]` ≈ `-ns` · `setting_sources=[]` ≈ `-nc` (sin CLAUDE.md) · sin `mcp_servers` ≈ `-ne`.

In [1]:
import os, json, logging, warnings
from pathlib import Path

logging.getLogger("anthropic").setLevel(logging.ERROR); warnings.filterwarnings("ignore")

from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))      # lee notebooks/.env si existe; no pisa variables ya exportadas

if "ANTHROPIC_API_KEY" not in os.environ:
    raise SystemExit("Falta ANTHROPIC_API_KEY en el entorno.")

MODEL = "claude-sonnet-5"
WS = (Path.cwd() if Path.cwd().name == "workspace" else Path("workspace")).resolve()  # contabilidad.csv (sintético) + .claude/skills/
PREGUNTA = '¿Cómo cambió el margen por proyecto de junio a julio, y por qué?'
SYSTEM = """Eres un analista financiero. Trabajas en el directorio actual, donde está ./contabilidad.csv
(separador ';', decimal con coma; columnas Period, AccountId, Debito, Credito, ProjectId, CostCenterName). No busques en otros directorios.
Reglas: ingreso = cuentas que empiezan por 4 (Credito - Debito); costo = cuentas 6 y 7 (Debito - Credito).
Margen = (ingreso - costo) / ingreso. Period 6 = junio, 7 = julio.
Verifica antes de responder. Salida: cifra por proyecto, el cálculo y la causa del cambio. Español, corto."""
SYSTEM_CORTO = 'Eres un analista financiero. Español, corto.'

from claude_agent_sdk import (query, ClaudeAgentOptions, AssistantMessage, ResultMessage,
                              TextBlock, ToolUseBlock, tool, create_sdk_mcp_server)

os.environ["CLAUDE_CODE_DISABLE_AUTO_MEMORY"] = "1"   # sin la memoria del usuario: el punto de partida es el modelo solo

SCHEMA = {
  "type": "object",
  "properties": {
    "proyectos": {
      "type": "array",
      "items": {
        "type": "object",
        "properties": {
          "proyecto": {
            "type": "string"
          },
          "margen_junio": {
            "type": "number"
          },
          "margen_julio": {
            "type": "number"
          },
          "cambio_pp": {
            "type": "number"
          },
          "causa": {
            "type": "string"
          }
        },
        "required": [
          "proyecto",
          "margen_junio",
          "margen_julio",
          "cambio_pp",
          "causa"
        ]
      }
    },
    "resumen": {
      "type": "string"
    }
  },
  "required": [
    "proyectos",
    "resumen"
  ]
}   # salida estructurada del MVP (paso 8)

async def ask(prompt=PREGUNTA, **opciones):
    """Corre una query y muestra tool calls, texto y costo."""
    opts = ClaudeAgentOptions(model=MODEL, cwd=str(WS), max_turns=25, stderr=lambda _: None, **opciones)
    async for m in query(prompt=prompt, options=opts):
        if isinstance(m, AssistantMessage):
            for b in m.content:
                if isinstance(b, ToolUseBlock):
                    arg = b.input.get("command") or b.input.get("file_path") or b.input.get("query") or b.input.get("skill") or b.input.get("pattern") or ""
                    print(f"→ {b.name}: {str(arg)[:90]}")
                elif isinstance(b, TextBlock) and b.text.strip():
                    print(b.text.strip())
        elif isinstance(m, ResultMessage):
            print(f"\n[{m.num_turns} turnos · ${m.total_cost_usd:.3f} · {m.duration_ms / 1000:.0f}s]")
            return m

## 1 · Solo el modelo
Sin tools, sin skills, sin contexto, sin system prompt.

In [2]:
await ask(tools=[], skills=[], setting_sources=[], system_prompt=None)

No tengo acceso a datos financieros, reportes o información específica de tu empresa o proyectos. Para poder responder a tu pregunta sobre cómo cambió el margen por proyecto de junio a julio, necesitaría que me proporciones:

1. **Los datos de margen** de junio y julio por proyecto (ingresos, costos, o el margen ya calculado)
2. **Información de contexto** que pueda explicar variaciones, como:
   - Cambios en costos (materiales, personal, proveedores)
   - Variaciones en precios de venta
   - Cambios en volumen o alcance del proyecto
   - Factores estacionales
   - Problemas operativos o retrasos

Si me compartes esta información (puede ser una tabla, archivo, o simplemente los números), con gusto te ayudo a:
- Calcular las variaciones de margen por proyecto
- Identificar tendencias o patrones
- Analizar posibles causas de los cambios
- Preparar un resumen o visualización de los resultados

¿Tienes esos datos disponibles para compartir?

[1 turnos · $0.005 · 46s]


ResultMessage(subtype='success', duration_ms=45894, duration_api_ms=77978, is_error=False, num_turns=1, session_id='6ac94565-b611-4d48-9840-ffb26bf357d1', stop_reason='end_turn', total_cost_usd=0.005064000000000001, usage={'input_tokens': 204, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'output_tokens': 364, 'output_tokens_details': {'thinking_tokens': 0}, 'server_tool_use': {'web_search_requests': 0, 'web_fetch_requests': 0}, 'service_tier': 'standard', 'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'inference_geo': 'global', 'iterations': [{'input_tokens': 204, 'output_tokens': 364, 'cache_read_input_tokens': 0, 'cache_creation_input_tokens': 0, 'cache_creation': {'ephemeral_5m_input_tokens': 0, 'ephemeral_1h_input_tokens': 0}, 'type': 'message'}], 'speed': 'standard'}, result='No tengo acceso a datos financieros, reportes o información específica de tu empresa o proyectos. Para poder responder a tu pregunta sobre cómo cambió e

## 2 · + system prompt
Objetivo, reglas y formato. Todavía no puede leer el archivo.

In [3]:
await ask(tools=[], skills=[], setting_sources=[], system_prompt=SYSTEM)

Voy a examinar el archivo de contabilidad para calcular los márgenes por proyecto.

**herramienta:** bash
```bash
cd . && head -5 contabilidad.csv && echo "---" && wc -l contabilidad.csv
```

*(Resultado esperado: ver estructura y tamaño del archivo)*

**herramienta:** bash
```bash
python3 << 'EOF'
import csv
from collections import defaultdict

# Leer CSV
rows = []
with open('contabilidad.csv', 'r', encoding='utf-8') as f:
    reader = csv.DictReader(f, delimiter=';')
    for row in reader:
        rows.append(row)

print("Columnas:", rows[0].keys())
print("Total filas:", len(rows))
print("Periodos únicos:", set(r['Period'] for r in rows))
print("Proyectos únicos:", set(r['ProjectId'] for r in rows))
print("Muestra AccountId:", set(r['AccountId'][:1] for r in rows))
EOF
```

*(Necesito ver la salida real antes de continuar con el cálculo)*

---

**Nota:** No tengo acceso directo a ejecutar comandos en este entorno de conversación — necesito que me confirmes si puedo proceder ejecutand

ResultMessage(subtype='success', duration_ms=84307, duration_api_ms=85441, is_error=False, num_turns=1, session_id='cd64a4a3-246e-4e6e-92a9-4643af620ee9', stop_reason='end_turn', total_cost_usd=0.00943, usage={'input_tokens': 432, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'output_tokens': 755, 'output_tokens_details': {'thinking_tokens': 19}, 'server_tool_use': {'web_search_requests': 0, 'web_fetch_requests': 0}, 'service_tier': 'standard', 'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'inference_geo': 'global', 'iterations': [{'input_tokens': 432, 'output_tokens': 755, 'cache_read_input_tokens': 0, 'cache_creation_input_tokens': 0, 'cache_creation': {'ephemeral_5m_input_tokens': 0, 'ephemeral_1h_input_tokens': 0}, 'type': 'message'}], 'speed': 'standard'}, result='Voy a examinar el archivo de contabilidad para calcular los márgenes por proyecto.\n\n**herramienta:** bash\n```bash\ncd . && head -5 contabilidad.csv && echo "---"

## 3 · + tool Bash
`tools` = qué existe. `allowed_tools` = qué corre sin preguntar.

In [4]:
await ask(system_prompt=SYSTEM, skills=[], setting_sources=[],
          tools=["Bash"], allowed_tools=["Bash"])

→ Bash: cd "$(pwd)" && ls -la ./contabilidad.csv && wc -l ./contabilidad.csv && head -5 ./contabil


→ Bash: cd "$(pwd)" && python3 - <<'EOF'
import csv
from collections import defaultdict

periods =


→ Bash: cd "$(pwd)" && python3 - <<'EOF'
import csv
from collections import defaultdict

def to_fl


→ Bash: cd "$(pwd)" && python3 - <<'EOF'
import csv
from collections import defaultdict

def to_fl


## Margen por proyecto: Junio → Julio

**Cálculo:** Margen = (Ingreso − Costo) / Ingreso · Ingreso = cuentas "4" (Créd−Déb) · Costo = cuentas "6"/"7" (Déb−Créd)

| Proyecto | Margen Jun | Margen Jul | Cambio | Causa principal |
|---|---|---|---|---|
| 1030 | 30.6% | 33.0% | ▲ +2.4 pp | Ingreso subió 3.8%; costos casi planos |
| 1045 | 33.7% | 11.5% | ▼ −22.2 pp | Costo de mercancía vendida +$57.5M y sueldos +$19M; solo compensado parcialmente por baja en arriendo/mantenimiento |
| 2210 | 38.0% | 48.0% | ▲ +10.0 pp | Ingreso creció 48.6% (a $260M) mucho más que el costo (+24.6%, pese a mantenimiento vehículos +$25.9M) |
| 2235 | 30.0% | 22.0% | ▼ −8.0 pp | Ingreso casi estable (+2.5%) pero mantenimiento vehículos +$14.8M sube el costo total |
| 3310 | 22.7% | 19.5% | ▼ −3.2 pp | Ingreso bajó 2.8% mientras el costo subió levemente |
| 3322 | 30.0% | 29.7% | ▼ −0.4 pp | Prácticamente sin cambio (ingreso y costo bajaron proporcionalmente) |
| 4410 | 30.1% | 28.0% | ▼ −2.1 pp | Mantenimient

ResultMessage(subtype='success', duration_ms=44905, duration_api_ms=39515, is_error=False, num_turns=5, session_id='e6f166b8-b5ca-496d-b273-6c46349d8a45', stop_reason='end_turn', total_cost_usd=0.0636689, usage={'input_tokens': 10, 'cache_creation_input_tokens': 9805, 'cache_read_input_tokens': 25182, 'output_tokens': 3410, 'output_tokens_details': {'thinking_tokens': 290}, 'server_tool_use': {'web_search_requests': 0, 'web_fetch_requests': 0}, 'service_tier': 'standard', 'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 9805}, 'inference_geo': 'global', 'iterations': [{'input_tokens': 2, 'output_tokens': 1040, 'cache_read_input_tokens': 8274, 'cache_creation_input_tokens': 1531, 'cache_creation': {'ephemeral_5m_input_tokens': 1531, 'ephemeral_1h_input_tokens': 0}, 'type': 'message'}], 'speed': 'standard'}, result='## Margen por proyecto: Junio → Julio\n\n**Cálculo:** Margen = (Ingreso − Costo) / Ingreso · Ingreso = cuentas "4" (Créd−Déb) · Costo = cuentas

## 4 · + Read, Glob, Grep
Mira el archivo antes de calcular.

In [5]:
await ask(system_prompt=SYSTEM, skills=[], setting_sources=[],
          tools=["Bash", "Read", "Glob", "Grep"], allowed_tools=["Bash", "Read", "Glob", "Grep"])

→ Bash: cd . && head -5 contabilidad.csv && echo "---" && wc -l contabilidad.csv


→ Bash: cd . && python3 << 'EOF'
import csv

rows=[]
with open('contabilidad.csv', encoding='utf-8


→ Bash: cd . && python3 << 'EOF'
import csv
from collections import defaultdict

def parse_num(s):


→ Bash: cd . && python3 << 'EOF'
import csv
from collections import defaultdict

def parse_num(s):


→ Bash: cd . && python3 << 'EOF'
import csv
from collections import defaultdict

def parse_num(s):


## Margen por proyecto: Junio → Julio

**Fórmula:** Margen = (Ingreso[cta 4] − Costo[cta 6+7]) / Ingreso[cta 4]

| Proyecto (Centro Costo) | Margen Jun | Margen Jul | Δ pp | Causa principal |
|---|---|---|---|---|
| 1030 (WAREHOUSE) | 30,6% | 33,0% | **+2,4** | Ingreso subió (+$11,3M almacenamiento) más que el costo neto; cayó arriendo bodega (-$9,3M) compensando alza en mant. vehículos (+$11,3M). |
| 1045 (WAREHOUSE) | 33,7% | 11,5% | **-22,2** | Costo de mercancía vendida se disparó (+$57,5M) y sueldos (+$19M), mientras el ingreso casi no creció. |
| 2210 (FIRST MILE) | 38,0% | 48,0% | **+10,0** | Ingreso creció fuerte (+$85M servicio transporte) muy por encima del alza en mant. vehículos (+$25,9M). |
| 2235 (FIRST MILE) | 30,0% | 22,0% | **-8,0** | Ingreso casi plano (+$3M) pero mant. vehículos subió +$14,8M. |
| 3310 (LAST MILE COL) | 22,7% | 19,5% | **-3,2** | Ingreso bajó (-$10M) y combustibles subieron (+$20,6M), pese a menor mant. vehículos. |
| 3322 (LAST MILE COL) | 30,0% | 2

ResultMessage(subtype='success', duration_ms=217208, duration_api_ms=288649, is_error=False, num_turns=6, session_id='9b40f838-306d-477e-bd5d-a7c177c7cdae', stop_reason='end_turn', total_cost_usd=0.08371590000000001, usage={'input_tokens': 12, 'cache_creation_input_tokens': 12745, 'cache_read_input_tokens': 45092, 'output_tokens': 4181, 'output_tokens_details': {'thinking_tokens': 1212}, 'server_tool_use': {'web_search_requests': 0, 'web_fetch_requests': 0}, 'service_tier': 'standard', 'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 12745}, 'inference_geo': 'global', 'iterations': [{'input_tokens': 2, 'output_tokens': 1775, 'cache_read_input_tokens': 10613, 'cache_creation_input_tokens': 2132, 'cache_creation': {'ephemeral_5m_input_tokens': 2132, 'ephemeral_1h_input_tokens': 0}, 'type': 'message'}], 'speed': 'standard'}, result='## Margen por proyecto: Junio → Julio\n\n**Fórmula:** Margen = (Ingreso[cta 4] − Costo[cta 6+7]) / Ingreso[cta 4]\n\n| Proyecto

## 5 · + WebSearch
Una pregunta que no está en el CSV.

In [2]:
await ask("¿Qué registra la cuenta 7205 del PUC colombiano? Busca en la web y cita la fuente en una frase.",
          system_prompt=SYSTEM, skills=[], setting_sources=[],
          tools=["Bash", "Read", "Glob", "Grep", "WebSearch"],
          allowed_tools=["Bash", "Read", "Glob", "Grep", "WebSearch"])

→ WebSearch: cuenta 7205 PUC colombiano costos de producción


→ WebSearch: "7205" puc.com.co cesantías mano de obra directa


La cuenta **7205** del PUC colombiano corresponde a **"Jornales"**, dentro del grupo 72 – Mano de Obra Directa (clase 7, Costos de producción u operación); registra el valor pagado por jornales/salarios del personal vinculado directamente al proceso productivo.

Fuente: [Cuenta 72 Mano de obra directa - puc.com.co](https://puc.com.co/72)

[3 turnos · $0.079 · 30s]


ResultMessage(subtype='success', duration_ms=29763, duration_api_ms=30548, is_error=False, num_turns=3, session_id='22edf94c-147f-46db-9a70-a6e418150f87', stop_reason='end_turn', total_cost_usd=0.0793782, usage={'input_tokens': 6, 'cache_creation_input_tokens': 10580, 'cache_read_input_tokens': 17611, 'output_tokens': 439, 'output_tokens_details': {'thinking_tokens': 153}, 'server_tool_use': {'web_search_requests': 0, 'web_fetch_requests': 0}, 'service_tier': 'standard', 'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 10580}, 'inference_geo': 'global', 'iterations': [{'input_tokens': 2, 'output_tokens': 146, 'cache_read_input_tokens': 9415, 'cache_creation_input_tokens': 1165, 'cache_creation': {'ephemeral_5m_input_tokens': 1165, 'ephemeral_1h_input_tokens': 0}, 'type': 'message'}], 'speed': 'standard'}, result='La cuenta **7205** del PUC colombiano corresponde a **"Jornales"**, dentro del grupo 72 – Mano de Obra Directa (clase 7, Costos de producción u 

## 6 · + skill
System prompt corto: las reglas viven en `workspace/.claude/skills/margen-por-proyecto/SKILL.md`.

In [7]:
print((WS / ".claude/skills/margen-por-proyecto/SKILL.md").read_text()[:600], "…\n")
await ask(system_prompt=SYSTEM_CORTO, skills=["margen-por-proyecto"],
          tools=["Bash", "Read", "Glob", "Grep", "Skill"], allowed_tools=["Bash", "Read", "Glob", "Grep", "Skill"])

---
name: margen-por-proyecto
description: Calcula el margen por proyecto y su cambio entre meses a partir de contabilidad.csv (PUC colombiano). Úsala cuando pregunten por margen, rentabilidad o utilidad por proyecto.
---

# Margen por proyecto

## Datos
`contabilidad.csv`: separador `;`, decimal con coma. Columnas clave: `Period` (6 = junio, 7 = julio), `AccountId`, `Debito`, `Credito`, `ProjectId`, `CostCenterName`.

## Reglas
- Ingreso = cuentas que empiezan por `4`: `Credito - Debito`.
- Costo = cuentas que empiezan por `6` o `7`: `Debito - Credito`.
- Margen = (ingreso - costo) / ingreso. …



→ Skill: margen-por-proyecto


→ Bash: find ~/charlas/2026-agents-bogota -iname "contabilidad.csv"


→ Bash: uv run --with pandas python3 - <<'PY'
import pandas as pd
df = pd.read_csv("contabilidad.c


→ Bash: uv run --with pandas python3 - <<'PY'
import pandas as pd
df = pd.read_csv("contabilidad.c


## Margen por proyecto: junio → julio

| Proyecto | Jun | Jul | Cambio |
|---|---|---|---|
| **1045** | 33.7% | 11.5% | **-22.2 pp** |
| 2235 | 30.0% | 22.0% | -8.0 pp |
| 3310 | 22.7% | 19.5% | -3.2 pp |
| 4410 | 30.1% | 28.0% | -2.1 pp |
| 3322 | 30.0% | 29.7% | -0.3 pp |
| 1030 | 30.6% | 33.0% | +2.4 pp |
| **2210** | 38.0% | 48.0% | **+10.0 pp** |

*(Proyectos 0 y 5001 sin ingreso válido → margen no calculable, no se incluyen.)*

**Causas principales:**

- **Proyecto 1045 (mayor caída, -22.2pp):** el ingreso creció poco (223M→230M), pero el **costo de mercancía vendida casi se duplicó** (+57.5M) y **sueldos y salarios subieron +19M** (probablemente refuerzo de personal). El ahorro en arriendo bodega (-7.3M) y mantenimiento vehículos (-15M) no alcanzó a compensar.

- **Proyecto 2210 (mayor mejora, +10pp):** el ingreso saltó de 175M a 260M (+49%) mientras los costos crecieron mucho menos proporcionalmente —solo mantenimiento vehículos subió (+25.9M)—, así que el mayor volumen de fact

ResultMessage(subtype='success', duration_ms=141964, duration_api_ms=120865, is_error=False, num_turns=6, session_id='b12f0475-bbc9-40c8-9313-4042d1f1c331', stop_reason='end_turn', total_cost_usd=0.060336100000000004, usage={'input_tokens': 10, 'cache_creation_input_tokens': 12641, 'cache_read_input_tokens': 41888, 'output_tokens': 1932, 'output_tokens_details': {'thinking_tokens': 114}, 'server_tool_use': {'web_search_requests': 0, 'web_fetch_requests': 0}, 'service_tier': 'standard', 'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 12641}, 'inference_geo': 'global', 'iterations': [{'input_tokens': 2, 'output_tokens': 668, 'cache_read_input_tokens': 11372, 'cache_creation_input_tokens': 1269, 'cache_creation': {'ephemeral_5m_input_tokens': 1269, 'ephemeral_1h_input_tokens': 0}, 'type': 'message'}], 'speed': 'standard'}, result='## Margen por proyecto: junio → julio\n\n| Proyecto | Jun | Jul | Cambio |\n|---|---|---|---|\n| **1045** | 33.7% | 11.5% | **-2

## 7 · + CLAUDE.md (contexto del proyecto)
`setting_sources=["project"]` carga `CLAUDE.md` y `.claude/` del cwd.

In [8]:
(WS / "CLAUDE.md").write_text("""# Contexto del proyecto
Python: usa `uv run --with pandas python3 -` (pandas no está en el Python del sistema).
Respuesta: tabla por proyecto (junio, julio, cambio en pp; máximo 8 filas), la causa principal en 2 frases, sin introducción ni cierre.
""")
await ask(system_prompt=SYSTEM_CORTO, setting_sources=["project"], skills=["margen-por-proyecto"],
          tools=["Bash", "Read", "Glob", "Grep", "Skill"], allowed_tools=["Bash", "Read", "Glob", "Grep", "Skill"])

→ Skill: margen-por-proyecto


→ Glob: **/contabilidad.csv


→ Bash: uv run --with pandas python3 - <<'PY'
import pandas as pd
df = pd.read_csv("contabilidad.c


→ Bash: uv run --with pandas python3 - <<'PY'
import pandas as pd
df = pd.read_csv("contabilidad.c


→ Bash: uv run --with pandas python3 - <<'PY'
import pandas as pd
df = pd.read_csv("contabilidad.c


Margen por proyecto (junio → julio):

| Proyecto | Junio | Julio | Cambio (pp) |
|---|---|---|---|
| 1045 | 33.7% | 11.5% | -22.2 |
| 2235 | 30.0% | 22.0% | -8.0 |
| 3310 | 22.7% | 19.5% | -3.2 |
| 4410 | 30.1% | 28.0% | -2.1 |
| 3322 | 30.0% | 29.7% | -0.3 |
| 1030 | 30.6% | 33.0% | +2.4 |
| 2210 | 38.0% | 48.0% | +10.0 |

La mayor caída fue en el proyecto 1045 (-22.2 pp), explicada principalmente por un fuerte aumento en el costo de mercancía vendida (+$57.5M) en julio. En segundo lugar contribuyó el alza en sueldos y salarios (+$19M).

[7 turnos · $0.059 · 22s]


ResultMessage(subtype='success', duration_ms=22074, duration_api_ms=21662, is_error=False, num_turns=7, session_id='3869bfaf-efd9-4c53-8f12-776bf42642ec', stop_reason='end_turn', total_cost_usd=0.058733400000000005, usage={'input_tokens': 12, 'cache_creation_input_tokens': 12162, 'cache_read_input_tokens': 49742, 'output_tokens': 1734, 'output_tokens_details': {'thinking_tokens': 223}, 'server_tool_use': {'web_search_requests': 0, 'web_fetch_requests': 0}, 'service_tier': 'standard', 'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 12162}, 'inference_geo': 'global', 'iterations': [{'input_tokens': 2, 'output_tokens': 325, 'cache_read_input_tokens': 11503, 'cache_creation_input_tokens': 659, 'cache_creation': {'ephemeral_5m_input_tokens': 659, 'ephemeral_1h_input_tokens': 0}, 'type': 'message'}], 'speed': 'standard'}, result='Margen por proyecto (junio → julio):\n\n| Proyecto | Junio | Julio | Cambio (pp) |\n|---|---|---|---|\n| 1045 | 33.7% | 11.5% | -22.

## 8 · MVP: tool propia + salida estructurada + límites
Una tool in-process (MCP), `output_format` con JSON Schema, `max_budget_usd`. Lo que consume una Web UI.

In [9]:
@tool("guardar_hallazgo", "Guarda un hallazgo (proyecto, cambio en pp, causa). Llama una vez por proyecto con |cambio| > 5 pp.",
      {"proyecto": str, "cambio_pp": float, "causa": str})
async def guardar_hallazgo(args):
    (WS / "out").mkdir(exist_ok=True)
    with open(WS / "out/hallazgos.jsonl", "a") as f:
        f.write(json.dumps(args, ensure_ascii=False) + "\n")
    return {"content": [{"type": "text", "text": f"guardado {args['proyecto']}"}]}

mvp = create_sdk_mcp_server("analista", tools=[guardar_hallazgo])
(WS / "out/hallazgos.jsonl").unlink(missing_ok=True)

r = await ask(system_prompt=SYSTEM_CORTO, setting_sources=["project"], skills=["margen-por-proyecto"],
              mcp_servers={"analista": mvp},
              tools=["Bash", "Read", "Glob", "Grep", "Skill"],
              allowed_tools=["Bash", "Read", "Glob", "Grep", "Skill", "mcp__analista__guardar_hallazgo"],
              output_format={"type": "json_schema", "schema": SCHEMA},
              max_budget_usd=1.0)
print(json.dumps(r.structured_output, indent=2, ensure_ascii=False))
print("\nhallazgos.jsonl:\n" + (WS / "out/hallazgos.jsonl").read_text())

→ Skill: margen-por-proyecto


→ Bash: find ~/charlas/2026-agents-bogota/notebooks/workspace -iname "


→ Bash: uv run --with pandas python3 - <<'PY'
import pandas as pd
df = pd.read_csv("contabilidad.c


→ Bash: uv run --with pandas python3 - <<'PY'
import pandas as pd
df = pd.read_csv("contabilidad.c


→ Bash: uv run --with pandas python3 - <<'PY'
import pandas as pd
df = pd.read_csv("contabilidad.c


...

→ Bash: uv run --with pandas python3 - <<'PY'
import pandas as pd
df = pd.read_csv("contabilidad.c


→ mcp__analista__guardar_hallazgo: 


→ mcp__analista__guardar_hallazgo: 


→ mcp__analista__guardar_hallazgo: 


→ StructuredOutput: 

[13 turnos · $0.102 · 188s]
{
  "proyectos": [
    {
      "proyecto": "1045",
      "margen_junio": 33.7,
      "margen_julio": 11.5,
      "cambio_pp": -22.17,
      "causa": "Apareció costo de mercancía vendida (+$57.5M) inexistente en junio y subieron sueldos (+$19M), sin crecimiento proporcional del ingreso."
    },
    {
      "proyecto": "2235",
      "margen_junio": 30,
      "margen_julio": 22,
      "cambio_pp": -8.01,
      "causa": "El mantenimiento de vehículos subió $14.8M, superando ahorros en combustible y nómina."
    },
    {
      "proyecto": "3310",
      "margen_junio": 22.7,
      "margen_julio": 19.5,
      "cambio_pp": -3.19,
      "causa": "Variación menor, no supera el umbral de análisis detallado."
    },
    {
      "proyecto": "4410",
      "margen_junio": 30.1,
      "margen_julio": 28,
      "cambio_pp": -2.14,
      "causa": "Variación menor, no supera el umbral de análisis detallado."
    },
    {
      "proyecto": "3322",
      "m

Ocho celdas, ocho palancas: system prompt · tools · permisos · web · skills · contexto · tools propias · salida estructurada. El loop nunca lo escribiste.